In [ ]:
# Getting relevant libraries

import numpy as np 
import random
import matplotlib.pyplot as plt
import math
import matplotlib.cm as cm
import pickle
import os
import pandas as pd
import random
plt.rcParams['figure.figsize'] = [10, 7]
from matplotlib.colors import LinearSegmentedColormap
#import mpl_scatter_density # adds projection='scatter_density'
from scipy.stats import gaussian_kde
from scipy import optimize
from molmass import Formula
import csv
import re
import copy
import gc
import time
import molmass as ms
from tqdm import tqdm
from EmulatorLibrary import *
from Emulator1_0_2_Sept12_2025 import DualSaturationChemistry 

def random_char(y):
       return ''.join(random.choice(string.ascii_letters) for x in range(y))

def QFM_fO2(P, K):
    trans1 = 573 + (0.025 * P)
    if K > trans1:
        A = -25096.3
        B = 8.735
        D = 0.11
    else:
        A = -26455.3
        B = 10.344
        D = 0.092
    K += 273.15 # Celsius to Kelvin
    logfo2 = (A/K) + B + ((D * (P-1)) / K)
    return(logfo2)


# Compile once, use many times
_number_pattern = re.compile(r'[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?')

def pull_number(string):
    match = _number_pattern.search(string)
    return float(match.group()) if match else np.nan
    
def pull_letter(string, symbols = False):
    letters = ''
    accepted_chars = 'ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz'
    if symbols:
        accepted_chars += '_+=-,.<>?;[]{}\|!@#$%^&*() '
    for char in string:
        if char in accepted_chars:
            letters += char
    return letters

def concat_all(*args):
    return ''.join(str(arg) for arg in args)

def identify_binaries(digits):
    """Returns numpy array of all unique binaries possible given a number of digits"""
    if 2**digits > 1E7:
        return(str(f"imagine there are {2**digits} of combinations supplied here. We aren't paid enough to actually generate them :P"))
    digits = int(digits)
    binaries = np.zeros((2,digits))
    binaries[1,0] = 1
    
    for b in range(1,digits):
        new_binaries = np.copy(binaries)
        new_binaries[:,b] = 1
        binaries = np.append(binaries, new_binaries, axis = 0)
        
    return binaries.astype(int)

def squash_to_range(x, min_=0.1, max_=0.95):
    return x * (max_ - min_) + min_

def unsquash_from_range(x, min_=0.1, max_=0.95):
    return (x - min_) / (max_ - min_)

In [12]:
"""TORCH ML LOADING. MUST HAPPEN AFTER ABOVE BLOCK IS RUN (for some reason...)"""
import torch
from torch.utils.data import DataLoader, Dataset, random_split
from torch.autograd import Variable
from torch.nn import Linear, ReLU, CrossEntropyLoss, Sequential, Conv2d, MaxPool2d, Module, Softmax, Dropout, BCELoss, Sigmoid, MSELoss
from torch.optim import Adam, SGD, AdamW
import torch.nn as nn
import torch.nn.functional as F
from SaturationDataset import TensorDatasetNormalized, TensorDataset, TensorDatasetThree, TensorDatasetFour

In [ ]:

        
def relative_L1_loss(y_pred, y_true, mask=None, eps=1e-6):
    rel_error = (y_pred - y_true).abs() / (y_true.abs() + eps)
    if mask is not None:
        rel_error = rel_error * mask
        return rel_error.sum() / mask.sum().clamp(min=1)
    return rel_error.mean()

def symmetric_rel_l1(pred, target, eps=1e-6):
    denom = torch.clamp(torch.abs(pred) + torch.abs(target), min=eps)
    return torch.mean(torch.abs(pred - target) / denom)

def symmetric_rel_l2(pred, target, eps=1e-6):
    denom = torch.clamp(torch.abs(pred) + torch.abs(target), min=eps)
    return torch.mean((pred - target)**2 / denom)


In [14]:
Trainfilename = '102Datasets/MELTS_TrainsetSept25BatchCooling_subsetIntensive'
Testfilename = '102Datasets/MELTS_TestsetSept25BatchCooling_subsetIntensive'

#time.sleep(3600) # hr delay for data processing, add another five minutes to this time 

PTfO2min = torch.tensor([1,700,-5], device = 'cpu', dtype = torch.float)
PTfO2max = torch.tensor([10000,2000,5], device = 'cpu', dtype = torch.float)
min_tensor = torch.zeros(len(Elkeys)+3, device = 'cpu', dtype = torch.float)
min_tensor[:3] = PTfO2min
range_tensor = torch.ones(len(Elkeys)+3, device = 'cpu', dtype = torch.float)
range_tensor[:3] = PTfO2max - PTfO2min



class Normalizer:
    """Quick Normalzing object that holds minima and ranges for a dataset and converts into and out of [0,1]
    minmax normalization for interfacing with neural networks"""
    
    def __init__(self, min_tensor, range_tensor):
        assert len(min_tensor) == len(range_tensor), 'Minimum and range are not equal!'
        self.miner = min_tensor
        self.ranger = range_tensor
        
    def __len__(self):
        return len(self.miner)

    def denorm(self, x):
        return x * self.ranger + self.miner
    
    def norm(self, x):
        return (x - self.miner) / self.ranger

feature_path = Trainfilename+'features.npy'
binary_path  = Trainfilename+'binary_labels.npy'
label_path = Trainfilename+'labels.npy'
mole_path = Trainfilename+'molar_labels.npy'

featureMap = np.load(feature_path, mmap_mode='r')
binaryMap = np.load(binary_path, mmap_mode='r')
labelMap = np.load(label_path, mmap_mode='r')
moleMap = np.load(mole_path, mmap_mode='r')

min_tensor = torch.zeros(featureMap.shape[1], device = 'cpu', dtype = torch.float)
min_tensor[:3] = PTfO2min
range_tensor = torch.ones(featureMap.shape[1], device = 'cpu', dtype = torch.float)
range_tensor[:3] = PTfO2max - PTfO2min
normf = Normalizer(min_tensor=min_tensor, range_tensor=range_tensor)

Trainnormfeatures = normf.norm(torch.tensor(featureMap, device = 'cpu', dtype = torch.float))
del featureMap
gc.collect()

Trainbinaryfeatures = torch.tensor(binaryMap, device = 'cpu', dtype = torch.float)
del binaryMap
gc.collect()

Trainlabels = torch.tensor(labelMap, device = 'cpu', dtype = torch.float) @ torch.tensor(PxSpTransform[np.ix_(compositional_component_subset, compositional_component_subset)], dtype = torch.float)
del labelMap
gc.collect()

Trainmoles = torch.tensor(moleMap, device = 'cpu', dtype=torch.float).detach().numpy()
del moleMap
gc.collect()


def process_in_batches(Trainnormfeatures, Trainmoles, Trainlabels, batch_size=8192):

    # Precompute constant matrices as float32 tensors
    oxToEl_t = torch.tensor(oxToEl[:-1], dtype=torch.float32)
    MM_t = torch.tensor(MM[:-1, :-1], dtype=torch.float32)
    compToOx_t = torch.tensor(compToOx, dtype=torch.float32)
    oxToEl_full_t = torch.tensor(oxToEl, dtype=torch.float32)

    # Inverse only once
    oxToEl_inv = torch.linalg.inv(oxToEl_t)

    n_samples = Trainnormfeatures.size(0)

    bulk_wt_ox_chunks = []
    GTReconBulk_chunks = []

    for start in tqdm(range(0, n_samples, batch_size)):
        end = min(start + batch_size, n_samples)

        # === Bulk weights ===
        bulk_wt_ox = (
            (Trainnormfeatures[start:end, 3:] @ oxToEl_inv) @ MM_t
        )
        bulk_wt_ox = 100 * bulk_wt_ox / torch.sum(bulk_wt_ox, axis=1).reshape(-1, 1)
        bulk_wt_ox_chunks.append(bulk_wt_ox)

        # === Ground truth compositions ===
        GT_comps = torch.zeros(
            (end - start, label_indices['melts-liquid'][-1] + 1),
            dtype=torch.float32,
        )

        for phase in np.array(list(label_indices.keys())):
            moles = torch.tensor(
                Trainmoles[start:end, mass_phasedict[phase]].reshape(-1, 1),
                dtype=torch.float32,
            )
            if phase in compositionally_variable_phases:
                GT_comps[:, label_indices[phase]] = (
                    moles * Trainlabels[start:end, label_indices_comp[phase]].to(torch.float32)
                )
            else:
                GT_comps[:, label_indices[phase]] = moles

        # === Recon bulk oxides ===
        GTReconBulk_oxides = (
            ((GT_comps @ compToOx_t) @ oxToEl_full_t) @ oxToEl_inv
        ) @ MM_t
        GTReconBulk_oxides *= 100 / torch.sum(GTReconBulk_oxides, axis=1, keepdims=True)

        GTReconBulk_chunks.append(GTReconBulk_oxides)

    # Recombine all batches
    bulk_wt_ox = torch.cat(bulk_wt_ox_chunks, dim=0)
    GTReconBulk_oxides = torch.cat(GTReconBulk_chunks, dim=0)

    # === Compare rounded results ===
    train_mismatches = torch.unique(
        torch.where(
            torch.round(bulk_wt_ox, decimals=2) != torch.round(GTReconBulk_oxides, decimals=2)
        )[0]
    )

    return train_mismatches

train_mismatches = process_in_batches(Trainnormfeatures, Trainmoles, Trainlabels, batch_size=2**13)


print(train_mismatches.size())
#assert mismatches.size()[0] == 0

OOB = ((Trainlabels > 1).to(float) + (Trainlabels < 0).to(float)).to(bool)
badMap = torch.unique(torch.where(OOB)[0])
goodMap = torch.ones(Trainlabels.size()[0]).to(torch.bool)
#goodMap = torch.arange(Testlabels.size()[0])
#goodMap = goodMap[~torch.isin(goodMap, badMap)] # Exclude OOB IDs
goodMap[badMap] = False
goodMap[train_mismatches] = False


print(f"Train Features: {Trainnormfeatures.size()}, Binaries {Trainbinaryfeatures.size()}, labels: {Trainlabels.size()}")
Trainnormfeatures, Trainbinaryfeatures, Trainlabels, Trainmoles = Trainnormfeatures[goodMap], Trainbinaryfeatures[goodMap], Trainlabels[goodMap], Trainmoles[goodMap]
print(f"Train Features: {Trainnormfeatures.size()}, Binaries {Trainbinaryfeatures.size()}, labels: {Trainlabels.size()}")

# IncludeCr free assemblages with rare phases
Cr_in = Trainnormfeatures[:,-1] != 0 + torch.any(
    Trainbinaryfeatures[:,torch.tensor([mass_phasedict[phase] for phase in ['nepheline', 'leucite', 'analcime', 'alloy-solid', 'muscovite', 'k-feldspar']])] > 0.5, 
    dim = -1).to(torch.bool)
rhm_idx = torch.where(Trainbinaryfeatures[:,mass_phasedict['rhm-oxide']] > 0.5)[0]
Cr_in[rhm_idx[torch.randperm(len(rhm_idx))[:(len(rhm_idx)//3)]]] = True # Add back 1/3rd of rhm-oxides. 

Cr_out = (Trainnormfeatures[:,-1] == 0).to(torch.bool) 
    

print(f"Chrome in Training: {Cr_in.sum()}, Chrome Absent in Training: {Cr_out.sum()}")

binary_train_set_Cr = TensorDataset(features=Trainnormfeatures[Cr_in], labels=Trainbinaryfeatures[Cr_in])
full_train_set_Cr = TensorDatasetFour(features=Trainnormfeatures[Cr_in], binarylabels=Trainbinaryfeatures[Cr_in], labels = Trainlabels[Cr_in], molelabels = Trainmoles[Cr_in])

binary_train_set_NoCr = TensorDataset(features=Trainnormfeatures[Cr_out], labels=Trainbinaryfeatures[Cr_out])
full_train_set_NoCr = TensorDatasetFour(features=Trainnormfeatures[Cr_out], binarylabels=Trainbinaryfeatures[Cr_out], labels = Trainlabels[Cr_out], molelabels = Trainmoles[Cr_out])


feature_path = Testfilename+'features.npy'
binary_path  = Testfilename+'binary_labels.npy'
label_path = Testfilename+'labels.npy'
mole_path = Testfilename+'molar_labels.npy'

featureMap = np.load(feature_path, mmap_mode='r')
binaryMap = np.load(binary_path, mmap_mode='r')
labelMap = np.load(label_path, mmap_mode='r')
moleMap = np.load(mole_path, mmap_mode ='r')

Testnormfeatures = normf.norm(torch.tensor(featureMap, device = 'cpu', dtype = torch.float))
del featureMap
gc.collect()

Testbinaryfeatures = torch.tensor(binaryMap, device = 'cpu', dtype = torch.float)
del binaryMap
gc.collect()

Testlabels = torch.tensor(labelMap, device = 'cpu', dtype = torch.float) @ torch.tensor(PxSpTransform[np.ix_(compositional_component_subset, compositional_component_subset)], dtype = torch.float)
del labelMap
gc.collect()

Testmoles = torch.tensor(moleMap, device = 'cpu', dtype=torch.float).detach().numpy()
del moleMap
gc.collect()


## --- Test split ---
bulk_wt_ox = (
    (Testnormfeatures[:, 3:] @ torch.linalg.inv(torch.tensor(oxToEl[:-1], dtype=torch.float32)))
    @ torch.tensor(MM[:-1, :-1], dtype=torch.float32)
)
bulk_wt_ox = 100 * bulk_wt_ox / torch.sum(bulk_wt_ox, axis=1).reshape(-1, 1)

GT_comps = torch.zeros(
    (Testnormfeatures.size()[0], label_indices['melts-liquid'][-1] + 1),
    dtype=torch.float32,
)

for phase in np.array(list(label_indices.keys())):
    if phase in compositionally_variable_phases:
        GT_comps[:, label_indices[phase]] = ((
            torch.tensor(Testmoles[:, mass_phasedict[phase]].reshape(-1, 1), dtype = torch.float32))
            * Testlabels[:, label_indices_comp[phase]].to(torch.float32)
        )
    else:
        GT_comps[:, label_indices[phase]] = ((
            torch.tensor(Testmoles[:, mass_phasedict[phase]].reshape(-1, 1), dtype = torch.float32))
        )

GTReconBulk_oxides = (
    ((GT_comps @ torch.tensor(compToOx, dtype=torch.float32))
     @ torch.tensor(oxToEl, dtype=torch.float32))
    @ torch.linalg.inv(torch.tensor(oxToEl[:-1], dtype=torch.float32))
) @ torch.tensor(MM[:-1, :-1], dtype=torch.float32)
GTReconBulk_oxides *= 100 / torch.sum(GTReconBulk_oxides, axis=1, keepdims=True)

test_mismatches = torch.unique(
    torch.where(torch.round(bulk_wt_ox, decimals = 2) != torch.round(GTReconBulk_oxides, decimals = 2))[0]
)
print(test_mismatches.size())
print(bulk_wt_ox.size())
#assert mismatches.size()[0] == 0, f'mismatch: {mismatches.size()[0]} out of bulk_wt_ox.size()[0]'




OOB = ((Testlabels > 1).to(float) + (Testlabels < 0).to(float)).to(bool)
badMap = torch.unique(torch.where(OOB)[0])
goodMap = torch.ones(Testlabels.size()[0]).to(torch.bool)
goodMap[badMap] = False
goodMap[test_mismatches] = False
print(f"Test Features: {Testnormfeatures.size()}, Binaries {Testbinaryfeatures.size()}, labels: {Testlabels.size()}")
Testnormfeatures, Testbinaryfeatures, Testlabels, Testmoles = Testnormfeatures[goodMap], Testbinaryfeatures[goodMap], Testlabels[goodMap], Testmoles[goodMap]
print(f"Test Features: {Testnormfeatures.size()}, Binaries {Testbinaryfeatures.size()}, labels: {Testlabels.size()}")

# IncludeCr free assemblages with rare phases
Cr_in = Testnormfeatures[:,-1] != 0 + torch.any(
    Testbinaryfeatures[:,torch.tensor([mass_phasedict[phase] for phase in ['nepheline', 'leucite', 'analcime', 'alloy-solid', 'muscovite', 'k-feldspar']])] > 0.5, 
    dim = -1).to(torch.bool)
rhm_idx = torch.where(Testbinaryfeatures[:,mass_phasedict['rhm-oxide']] > 0.5)[0]
Cr_in[rhm_idx[torch.randperm(len(rhm_idx))[:(len(rhm_idx)//3)]]] = True # Add back 1/3rd of rhm-oxides. 

Cr_out = (Testnormfeatures[:,-1] == 0).to(torch.bool) 
          
          
print(f"Chrome in Test: {Cr_in.sum()}, Chrome Absent in Test: {Cr_out.sum()}")





binary_test_set_Cr = TensorDataset(features=Testnormfeatures[Cr_in], labels=Testbinaryfeatures[Cr_in])
full_test_set_Cr = TensorDatasetFour(features=Testnormfeatures[Cr_in], binarylabels=Testbinaryfeatures[Cr_in], labels = Testlabels[Cr_in], molelabels = Testmoles[Cr_in])

binary_test_set_NoCr = TensorDataset(features=Testnormfeatures[Cr_out], labels=Testbinaryfeatures[Cr_out])
full_test_set_NoCr = TensorDatasetFour(features=Testnormfeatures[Cr_out], binarylabels=Testbinaryfeatures[Cr_out], labels = Testlabels[Cr_out], molelabels = Testmoles[Cr_out])







100%|██████████| 461/461 [00:19<00:00, 23.44it/s]


torch.Size([2130])
Train Features: torch.Size([3773962, 14]), Binaries torch.Size([3773962, 20]), labels: torch.Size([3773962, 58])
Train Features: torch.Size([3769273, 14]), Binaries torch.Size([3769273, 20]), labels: torch.Size([3769273, 58])
Chrome in Training: 1528542, Chrome Absent in Training: 2708801
torch.Size([39])
torch.Size([52781, 11])
Test Features: torch.Size([52781, 14]), Binaries torch.Size([52781, 20]), labels: torch.Size([52781, 58])
Test Features: torch.Size([52616, 14]), Binaries torch.Size([52616, 20]), labels: torch.Size([52616, 58])
Chrome in Test: 18275, Chrome Absent in Test: 37874


In [ ]:
"""HYBRID NET Training loop for Binary Phase Saturation Model"""

criterion = nn.BCEWithLogitsLoss()  # suitable for multi-label classification
#criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weights.cuda())  # suitable for multi-label classification, weighting rare phases

device = 'cuda'

for i, (binary_train_set, binary_test_set) in enumerate([(binary_train_set_NoCr, binary_test_set_NoCr), (binary_train_set_Cr, binary_test_set_Cr)]):
    FullMELTS = DualSaturationChemistry().cuda()
    date = "Sept30" 
    modelname = "rhyoliteMELTS1.0.2Batch"
    DictFilePath=f"./{modelname}{['NoCr', 'Cr'][i]}_BinaryOnly_{date}.pt"
    #FullMELTS.load_state_dict(torch.load(DictFilePath), strict=False) #Warm Start
    #ictFilePath=f'./{modelname}_BinaryOnly0.0025noise_{date}.pt'
    
    batch_size = 1024
    binary_train_loader = DataLoader(binary_train_set, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True)
    binary_test_loader = DataLoader(binary_test_set, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)
    
    for p in FullMELTS.parameters():
        p.requires_grad = True
    for p in FullMELTS.chem_heads.parameters():
        p.requires_grad = False
    for p in FullMELTS.mole_head.parameters():
        p.requires_grad = False

    train_losses = []
    test_losses = []
    long_test_losses = []
    valid_loss_min_binary = np.inf

    #EPOCHS = 50
    epoch = 0
    improve_record = [1,1]
    lrs = np.logspace(-8,-4,9).tolist() 
    #lrs = np.logspace(-7,-3,9).tolist() 

    lr = lrs.pop()
    wd = 0#1E-4
    optimizer = Adam(FullMELTS.parameters(), lr=lr, weight_decay=wd)

    # ---- Training loop ----
    #for epoch in range(EPOCHS):
    while lr > 2E-7:

        epoch += 1
        epoch_start = time.time()
        FullMELTS.train()
        running_train_loss = 0.0
        total_batches = len(binary_train_loader)

        print(f"\n--- Epoch {epoch+1} ---") #/{EPOCHS}

        for batch_idx, (x_batch, y_batch) in enumerate(tqdm(binary_train_loader, desc="Training", leave=False)):
            x_batch, y_batch = x_batch.to(device, non_blocking=True), y_batch.to(device, non_blocking=True)

            x_batch = x_batch + torch.randn_like(x_batch) * 0.0025 # Add Small Gaussian Noise to avoid overfitting during training

            optimizer.zero_grad()
            logits = FullMELTS.forward_binaries(x_batch)
            loss = criterion(logits, y_batch)
            loss.backward()
            optimizer.step()

            running_train_loss += loss.item() * x_batch.size(0)

            if batch_idx % 200 == 0:
                percent_done = 100 * batch_idx / total_batches
                train_losses.append(loss.item())
                print(f"[{percent_done:>5.1f}%] Batch {batch_idx:>5d} Loss: {loss.item():.4f}")

        avg_train_loss = running_train_loss / len(binary_train_set)


        # ---- Evaluation ----
        FullMELTS.eval()
        running_test_loss = 0.0
        with torch.no_grad():
            for batch_idx, (x_batch, y_batch) in enumerate(binary_test_loader):
                x_batch, y_batch = x_batch.to(device, non_blocking=True), y_batch.to(device, non_blocking=True)
                logits = FullMELTS.forward_binaries(x_batch)
                loss = criterion(logits, y_batch)
                running_test_loss += loss.item() * x_batch.size(0)
                if batch_idx % 50 == 0:
                    long_test_losses.append(loss.item())

        avg_test_loss = running_test_loss / len(binary_test_set)
        test_losses.append(avg_test_loss)
        print(f"Running Saturation Loss: {round(running_test_loss,4)}")
        if avg_test_loss <= valid_loss_min_binary:
            torch.save(FullMELTS.state_dict(), DictFilePath)
            print('\tValidation loss decreased ({:.6f} --> {:.6f}).  Saving model ...'.format(valid_loss_min_binary, avg_test_loss))
            valid_loss_min_binary = avg_test_loss
            improve_record.append(1)
        else:
            improve_record.append(0)

        print(f"Epoch {epoch} | Train Loss: {avg_train_loss:.6f} | Test Loss: {avg_test_loss:.6f}")#/{EPOCHS}
        print(f"[TIMER] Epoch time: {time.time() - epoch_start:.2f} seconds")
        if np.sum(improve_record[-3:]) == 0:
            improve_record = [1,1]
            lr = lrs.pop()
            """if wd > 5*lr:
                wd = 5*lr"""
            print(f"No Improvement in 3 epochs. New LR: {lr}. New Weight Decay: {wd}")
            optimizer = Adam(FullMELTS.parameters(), lr=lr, weight_decay=wd)

    # ---- Plotting ----
    plt.figure(figsize=(8, 5))
    plt.plot((epoch/len(train_losses))*(np.arange(len(train_losses))+1),train_losses, label='Train Loss')
    plt.plot((epoch/len(long_test_losses))*(np.arange(len(long_test_losses))+1), long_test_losses, label='Test Loss')
    plt.xlabel("Epoch")
    plt.ylabel("Loss (BCEWithLogits)")
    plt.title("Phase Saturation Training and Test Loss")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(f"{modelname}_BinaryPhaseSatTrain_{date}.jpg", dpi = 256)
    plt.show()

    """Histograms of Binary Phase Saturation Probabilities"""

    directories = [f'{modelname}Binary_Phase_Saturation_Histograms_{date}_TRAIN',f'{modelname}Binary_Phase_Saturation_Histograms_{date}_TEST']
    for i, histogram_directory in enumerate(directories):

        if not os.path.exists(histogram_directory):
            os.makedirs(histogram_directory)
        if len([binary_train_set, binary_test_set][i]) > 500000:
            subset = np.random.choice(np.arange(0, len([binary_train_set, binary_test_set][i])), size=500000, replace=False)
        else:
            subset = np.arange(0, len([binary_train_set, binary_test_set][i]))

        Xtest, Ytest = ([binary_train_set, binary_test_set][i])[subset.tolist()]
        with torch.no_grad():
            Y_hat_test = torch.sigmoid(FullMELTS.forward_binaries(Xtest.to('cuda')))
            Y_hat_test = Y_hat_test.detach().cpu().numpy()
            Xtest = Xtest.detach().numpy()
            Ytest = Ytest.detach().numpy()
        gc.collect()
        with open(histogram_directory+'/PRstats.txt', 'w'): # Create blank file
            pass


        for i, phase in enumerate(list(label_indices.keys())):
            realPos = (Ytest[:,i] > 0.5)
            predPos = (Y_hat_test[:,i] > 0.5).astype(float)
            precision = Ytest[predPos.astype(bool),i].sum()/np.sum(predPos)
            recall = Y_hat_test[realPos.astype(bool),i].sum()/np.sum(realPos)
            with open(histogram_directory+'/PRstats.txt', 'a') as File: #Record Stats
                File.write(f"Model {phase} Precision of positive prediction: {round(100*precision,2)}\n")
                File.write(f"Model {phase} Recall of dataset positives : {round(100*recall,2)}%\n")
            print(f"Model {phase} Precision of positive prediction: {round(100*precision,2)}% ")
            print(f"Model {phase} Recall of dataset positives : {round(100*recall,2)}%")
            plt.hist(Y_hat_test[realPos.astype(bool),i], bins=30, alpha=0.5, color = 'blue', label=f'{phase} Present', density=True, log = True)
            plt.hist(Y_hat_test[~(realPos.astype(bool)),i], bins=30, alpha=0.5, color = 'red', label=f'{phase} Absent', density=True, log = True)
            #plt.axvline(x=3, color='r', linestyle='dashed', linewidth=1)
            plt.legend()
            plt.xlabel("Probability")
            plt.ylabel("Normalized Frequency (Log Scale)")
            plt.title(f"NN {phase} Saturation Probabilities\nPresent in {round(100*realPos.sum()/len(Ytest),2)}% of Dataset")
            plt.tight_layout()
            plt.savefig(histogram_directory+f"/{phase}_Saturation_Probability_Histogram")
            plt.show()


--- Epoch 2 ---


Training:   0%|          | 5/2646 [00:11<1:18:26,  1.78s/it]

[  0.0%] Batch     0 Loss: 0.6929


Training:   8%|▊         | 205/2646 [00:18<01:33, 26.09it/s]

[  7.6%] Batch   200 Loss: 0.3556


...

Training:   8%|▊         | 203/2646 [00:15<01:48, 22.45it/s]

[  7.6%] Batch   200 Loss: 0.0521


Training:  15%|█▌        | 405/2646 [00:23<01:21, 27.52it/s]

[ 15.1%] Batch   400 Loss: 0.0497


Training:  22%|██▏       | 588/2646 [00:30<01:29, 22.89it/s]

In [ ]:
#Chem and Mole head training. NoBulk, frozen encodings
#torch.autograd.set_detect_anomaly(False)

weight_dict_both = {
    'olivine': 3,
    'nepheline': 15,
    'leucite': 10,
    'alloy-solid': 5,
    'muscovite': 3,
    'k-feldspar': 5
}
# This overprints the above weights, does not apply to NoCr model
weight_dict_Cr = {
    'rhm-oxide': 5
}

binWeightsNoCr = torch.ones(len(list(label_indices.keys())), dtype = torch.float32).reshape(1,-1)
binWeightsCr = torch.ones(len(list(label_indices.keys())), dtype = torch.float32).reshape(1,-1)
compWeightsNoCr = torch.ones(label_indices_comp['melts-liquid'][-1]+1, dtype = torch.float32).reshape(1,-1)
compWeightsCr = torch.ones(label_indices_comp['melts-liquid'][-1]+1, dtype = torch.float32).reshape(1,-1)
for phase, W in weight_dict_both.items():
    binWeightsCr[:,mass_phasedict[phase]] = W
    binWeightsNoCr[:,mass_phasedict[phase]] = W
    if phase in compositionally_variable_phases:
        compWeightsCr[:,comp_phasedict[phase]] = W
        compWeightsNoCr[:,comp_phasedict[phase]] = W
for phase, W in weight_dict_Cr.items():
    binWeightsCr[:,mass_phasedict[phase]] = W
    if phase in compositionally_variable_phases:
        compWeightsCr[:,comp_phasedict[phase]] = W


for i, (full_train_set, full_test_set) in enumerate([(full_train_set_NoCr, full_test_set_NoCr), (full_train_set_Cr, full_test_set_Cr)]):

    FullMELTS = DualSaturationChemistry().cuda()
    #date = "Sept30"
    #modelname = "rhyoliteMELTS1.0.2Fxtal"
    DictFilePath=f"./{modelname}{['NoCr', 'Cr'][i]}_BinaryOnly_{date}.pt"
    #DictFilePath=f'./{modelname}_ChemMoleL2_{date}.pt'
    
    FullMELTS.load_state_dict(torch.load(DictFilePath),strict = False)
    
    #date = "Sept22" + ['NoCr', 'Cr'][i]
    DictFilePath=f"./{modelname}{['NoCr', 'Cr'][i]}_ChemMoleL2_{date}.pt"
    
    
    
    batch_size = 1024

    device = 'cuda'

    binWeights = ([binWeightsNoCr, binWeightsCr][i]).to(device)
    compWeights = ([compWeightsNoCr, compWeightsCr][i]).to(device)
    
    train_loader = DataLoader(full_train_set, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True)
    test_loader = DataLoader(full_test_set, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)

    # All but mole heads: Put those on top.
    for p in FullMELTS.parameters():
        p.requires_grad = True
    #for p in FullMELTS.mole_head.parameters():
    #    p.requires_grad = False
    for p in FullMELTS.encoder.parameters():
        p.requires_grad = False
        
        
    criterion_sat = torch.nn.BCEWithLogitsLoss(weight = binWeights)
    criterion_chem = symmetric_rel_l2 #relative_L1_loss #y_pred, y_true, mask=None, eps=1e-6 #criterion_chem = F.mse_loss 
    criterion_mole = symmetric_rel_l2
    criterion_bulk = symmetric_rel_l2
    
    chem_alpha = 1 
    mole_alpha = 1
    bulk_alpha = 0 # For now, guassian noise in bulk
    

    train_losses = []
    test_losses = []
    long_test_losses = []
    valid_loss_min = np.inf


    epoch = 0
    improve_record = [1,1]
    lrs = np.logspace(-7,-3,9).tolist()
    lr = lrs.pop()
    wd = 1E-5
    optimizer = AdamW(filter(lambda p: p.requires_grad, FullMELTS.parameters()), lr=lr, weight_decay=wd)

    # ---- Training loop ----
    #for epoch in range(EPOCHS):
    while (lr > 2E-7):
    #while lr > 2E-25:

        epoch += 1
        epoch_start = time.time()
        FullMELTS.train()
        running_train_loss = 0.0
        total_batches = len(train_loader)

        print(f"\n--- Epoch {epoch} ---") #/{EPOCHS}

        for batch_idx, (x_batch, b_batch, y_batch, m_batch) in enumerate(tqdm(train_loader, desc="Training", leave=False)):

            optimizer.zero_grad()

            x_batch, b_batch, y_batch, m_batch = x_batch.to(device, non_blocking=True), b_batch.to(device, non_blocking=True), y_batch.to(device, non_blocking=True), m_batch.to(device, non_blocking=True)
            
            bulk_zero_mask = (x_batch != 0).to(torch.float)
            x_batch = x_batch + torch.randn_like(x_batch) * 0.002 * bulk_zero_mask # Add Small Gaussian Noise to avoid overfitting during training

            logits, chem_preds, chem_zero_mask, mole_preds, bulk_preds = FullMELTS(x_batch, binaries=b_batch, NN_only = True)

            # Binary saturation loss
            loss_sat = criterion_sat(logits, b_batch)

            # Chemistry and mass losses: apply masks
            chem_loss_raw = criterion_chem(chem_preds, y_batch)
            mole_loss_raw = criterion_mole(mole_preds, m_batch)
            bulk_loss_raw = criterion_bulk(bulk_preds, x_batch[:,3:])
            mole_zero_mask = (torch.sigmoid(logits) > 0.5).to(torch.float).detach() # Only use binary preds for masking when those neurons are free
            mole_zero_mask = (b_batch > 0.5).to(torch.float).detach()
     
            chem_loss_masked = (chem_loss_raw * chem_zero_mask * compWeights).sum() / (chem_zero_mask * compWeights).sum().clamp(min=1)
            mole_loss_masked = (mole_loss_raw * mole_zero_mask * binWeights).sum() / (mole_zero_mask * binWeights).sum().clamp(min=1)
            bulk_loss_masked = (bulk_loss_raw * bulk_zero_mask[:,3:]).sum() / (bulk_zero_mask[:,3:]).sum().clamp(min=1)
            
            
            loss = loss_sat + chem_alpha*chem_loss_masked + mole_alpha*mole_loss_masked + bulk_alpha*bulk_loss_masked
            #print(f"Sat loss: {loss_sat.item()}, Chem loss: {chem_loss_masked.item()}")

            if not torch.isfinite(loss):
                print("Non-finite loss detected!")
                print(f"Sat loss: {loss_sat.item()}, Chem loss: {chem_loss_masked.item()}, Mole loss: {mole_loss_masked}, Bulk loss: {bulk_loss_masked}")
                continue

            loss.backward()


            optimizer.step()

            running_train_loss += loss.item() * x_batch.size(0)

            if batch_idx % 200 == 0:
                percent_done = 100 * batch_idx / total_batches
                train_losses.append(loss.item())
                print(f"[{percent_done:>5.1f}%] Batch {batch_idx:>5d} Loss: {loss.item():.4f}")


        avg_train_loss = running_train_loss / len(full_train_set)

        # ---- Evaluation ----
        FullMELTS.eval()
        running_test_loss = 0.0
        running_sat_loss = 0
        running_chem_loss = 0
        running_mole_loss = 0
        running_bulk_loss = 0
        
        with torch.no_grad():
            for batch_idx, (x_batch, b_batch, y_batch, m_batch) in enumerate(test_loader):
                x_batch, b_batch, y_batch, m_batch = x_batch.to(device, non_blocking=True), b_batch.to(device, non_blocking=True), y_batch.to(device, non_blocking=True), m_batch.to(device, non_blocking=True)
                logits, chem_preds, chem_zero_mask, mole_preds, bulk_preds = FullMELTS(x_batch, binaries=b_batch, NN_only = True)
                
                # Binary saturation loss
                loss_sat = criterion_sat(logits, b_batch)
                running_sat_loss += loss_sat.item()
                
                # Chemistry losses: apply masks
                chem_loss_raw = criterion_chem(chem_preds, y_batch)
                mole_loss_raw = criterion_mole(mole_preds, m_batch)
                bulk_loss_raw = criterion_bulk(bulk_preds, x_batch[:,3:])
                bulk_zero_mask = (x_batch[:,3:] != 0).to(torch.float)
                mole_zero_mask = (torch.sigmoid(logits) > 0.5).to(torch.float).detach()
                #mole_zero_mask = (b_batch > 0.5).to(torch.float)
                
                chem_loss_masked = (chem_loss_raw * chem_zero_mask).sum() / chem_zero_mask.sum().clamp(min=1)
                mole_loss_masked = (mole_loss_raw * mole_zero_mask).sum() / mole_zero_mask.sum().clamp(min=1)
                bulk_loss_masked = (bulk_loss_raw * bulk_zero_mask).sum() / bulk_zero_mask.sum().clamp(min=1)
                
                running_mole_loss += mole_loss_masked.item()
                running_chem_loss += chem_loss_masked.item()
                running_bulk_loss += bulk_loss_masked.item()
                
                # Total loss
                #loss = loss_sat + chem_alpha*chem_loss_masked
                #loss = mole_alpha*mole_loss_masked + bulk_alpha*bulk_loss_masked
                loss = loss_sat + chem_alpha*chem_loss_masked + mole_alpha*mole_loss_masked + bulk_alpha*bulk_loss_masked
                

                running_test_loss += loss.item() * x_batch.size(0)
                if batch_idx % 50 == 0:
                    long_test_losses.append(loss.item())


        print(f"Running Saturation Loss: {round(running_sat_loss,4)}\nRunning Chem Loss: {round(running_chem_loss,4)}")
        print(f"Running Molar Loss: {round(running_mole_loss,4)}\nRunning Bulk Loss: {round(running_bulk_loss,4)}")
        
        avg_test_loss = running_test_loss / len(full_test_set)
        test_losses.append(avg_test_loss)
        if avg_test_loss <= valid_loss_min:
            torch.save(FullMELTS.state_dict(), DictFilePath)
            print('\tValidation loss decreased ({:.6f} --> {:.6f}).  Saving model ...'.format(valid_loss_min, avg_test_loss))
            valid_loss_min = avg_test_loss
            improve_record.append(1)
        else:
            improve_record.append(0)

        print(f"Epoch {epoch+1} | Train Loss: {avg_train_loss:.6f} | Test Loss: {avg_test_loss:.6f}")#/{EPOCHS}
        print(f"[TIMER] Epoch time: {time.time() - epoch_start:.2f} seconds")
        if np.sum(improve_record[-3:]) == 0:
            improve_record = [1,1]
            lr = lrs.pop()
            #if wd > 5*lr:
            #    wd = 5*lr
            print(f"No Improvement in 3 epochs. New LR: {lr}. New Weight Decay: {wd}")
            optimizer = AdamW(filter(lambda p: p.requires_grad, FullMELTS.parameters()), lr=lr, weight_decay=wd)#Adam(FullMELTS.parameters(), lr=lr)#, weight_decay=wd)
        #Adam(FullMELTS.parameters(), lr=lr)#, weight_decay=wd)
        
        #break

In [ ]:
#FULL model with Bulk and custom phase weights
#torch.autograd.set_detect_anomaly(False)

weight_dict_both = {
    'olivine': 3,
    'nepheline': 15,
    'leucite': 10,
    'alloy-solid': 5,
    'muscovite': 3,
    'k-feldspar': 5
}
# This overprints the above weights, does not apply to NoCr model
weight_dict_Cr = {
    'rhm-oxide': 3
}

binWeightsNoCr = torch.ones(len(list(label_indices.keys())), dtype = torch.float32).reshape(1,-1)
binWeightsCr = torch.ones(len(list(label_indices.keys())), dtype = torch.float32).reshape(1,-1)
compWeightsNoCr = torch.ones(label_indices_comp['melts-liquid'][-1]+1, dtype = torch.float32).reshape(1,-1)
compWeightsCr = torch.ones(label_indices_comp['melts-liquid'][-1]+1, dtype = torch.float32).reshape(1,-1)
for phase, W in weight_dict_both.items():
    binWeightsCr[:,mass_phasedict[phase]] = W
    binWeightsNoCr[:,mass_phasedict[phase]] = W
    if phase in compositionally_variable_phases:
        compWeightsCr[:,comp_phasedict[phase]] = W
        compWeightsNoCr[:,comp_phasedict[phase]] = W
for phase, W in weight_dict_Cr.items():
    binWeightsCr[:,mass_phasedict[phase]] = W
    if phase in compositionally_variable_phases:
        compWeightsCr[:,comp_phasedict[phase]] = W


for i, (full_train_set, full_test_set) in enumerate([(full_train_set_NoCr, full_test_set_NoCr), (full_train_set_Cr, full_test_set_Cr)]):
    #if i == 0:
    #   continue
    FullMELTS = DualSaturationChemistry().cuda()
    #date = "Sept30" 
    #modelname = "rhyoliteMELTS1.0.2Fxtal"
    DictFilePath=f"./{modelname}{['NoCr', 'Cr'][i]}_ChemMoleL2_{date}.pt"

    FullMELTS.load_state_dict(torch.load(DictFilePath),strict = False)
    
    #date = "Sept25" + ['NoCr', 'Cr'][i]
    DictFilePath=f"./{modelname}{['NoCr', 'Cr'][i]}_FullL1_{date}.pt"
    
    
    
    batch_size = 1024

    device = 'cuda'

    binWeights = ([binWeightsNoCr, binWeightsCr][i]).to(device)
    compWeights = ([compWeightsNoCr, compWeightsCr][i]).to(device)
    
    train_loader = DataLoader(full_train_set, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True)
    test_loader = DataLoader(full_test_set, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)

    # All 
    #Training only mole and Chem heads
    for p in FullMELTS.parameters():
        p.requires_grad = False

    for p in FullMELTS.mole_head.parameters():
        p.requires_grad = True

    for p in FullMELTS.chem_heads.parameters():
        p.requires_grad = True
        
    criterion_sat = torch.nn.BCEWithLogitsLoss(weight = binWeights)
    criterion_chem = symmetric_rel_l1 #relative_L1_loss #y_pred, y_true, mask=None, eps=1e-6 #criterion_chem = F.mse_loss 
    criterion_mole = symmetric_rel_l1
    criterion_bulk = symmetric_rel_l1
    
    chem_alpha = 1 #tried 1/30 last time
    mole_alpha = 1
    bulk_alpha = 0
    

    train_losses = []
    test_losses = []
    long_test_losses = []
    valid_loss_min = np.inf


    epoch = 0
    improve_record = [1,1]
    lrs = np.logspace(-7,-3,9).tolist()
    
    lr = lrs.pop()
    wd = 1E-5
    optimizer = AdamW(filter(lambda p: p.requires_grad, FullMELTS.parameters()), lr=lr, weight_decay=wd)

    # ---- Training loop ----
    #for epoch in range(EPOCHS):
    while (lr > 2E-7):
    #while lr > 2E-25:

        epoch += 1
        epoch_start = time.time()
        FullMELTS.train()
        running_train_loss = 0.0
        total_batches = len(train_loader)

        print(f"\n--- Epoch {epoch} ---") #/{EPOCHS}

        for batch_idx, (x_batch, b_batch, y_batch, m_batch) in enumerate(tqdm(train_loader, desc="Training", leave=False)):

            optimizer.zero_grad()

            x_batch, b_batch, y_batch, m_batch = x_batch.to(device, non_blocking=True), b_batch.to(device, non_blocking=True), y_batch.to(device, non_blocking=True), m_batch.to(device, non_blocking=True)
            
            bulk_zero_mask = (x_batch != 0).to(torch.float)
            #x_batch = x_batch + torch.randn_like(x_batch) * 0.003 * bulk_zero_mask # Add Small Gaussian Noise to avoid overfitting during training

            logits, chem_preds, chem_zero_mask, mole_preds, bulk_preds = FullMELTS(x_batch, binaries=b_batch, NN_only = True)

            # Binary saturation loss
            loss_sat = criterion_sat(logits, b_batch)

            # Chemistry and mass losses: apply masks
            chem_loss_raw = criterion_chem(chem_preds, y_batch)
            mole_loss_raw = criterion_mole(mole_preds, m_batch)
            bulk_loss_raw = criterion_bulk(bulk_preds, x_batch[:,3:])
            #bulk_zero_mask = (x_batch[:,3:] != 0).to(torch.float)
            mole_zero_mask = (torch.sigmoid(logits) > 0.5).to(torch.float).detach() # Only use binary preds for masking when those neurons are free
            #mole_zero_mask = (b_batch > 0.5).to(torch.float).detach()
     
            chem_loss_masked = (chem_loss_raw * chem_zero_mask * compWeights).sum() / (chem_zero_mask * compWeights).sum().clamp(min=1)
            mole_loss_masked = (mole_loss_raw * mole_zero_mask * binWeights).sum() / (mole_zero_mask * binWeights).sum().clamp(min=1)
            bulk_loss_masked = (bulk_loss_raw * bulk_zero_mask).sum() / bulk_zero_mask.sum().clamp(min=1)
            
            
            loss = loss_sat + chem_alpha*chem_loss_masked + mole_alpha*mole_loss_masked + bulk_alpha*bulk_loss_masked
            #print(f"Sat loss: {loss_sat.item()}, Chem loss: {chem_loss_masked.item()}")

            if not torch.isfinite(loss):
                print("Non-finite loss detected!")
                print(f"Sat loss: {loss_sat.item()}, Chem loss: {chem_loss_masked.item()}, Mole loss: {mole_loss_masked}, Bulk loss: {bulk_loss_masked}")
                continue

            loss.backward()


            optimizer.step()

            running_train_loss += loss.item() * x_batch.size(0)

            if batch_idx % 200 == 0:
                percent_done = 100 * batch_idx / total_batches
                train_losses.append(loss.item())
                print(f"[{percent_done:>5.1f}%] Batch {batch_idx:>5d} Loss: {loss.item():.4f}")


        avg_train_loss = running_train_loss / len(full_train_set)

        # ---- Evaluation ----
        FullMELTS.eval()
        running_test_loss = 0.0
        running_sat_loss = 0
        running_chem_loss = 0
        running_mole_loss = 0
        running_bulk_loss = 0
        
        with torch.no_grad():
            for batch_idx, (x_batch, b_batch, y_batch, m_batch) in enumerate(test_loader):
                x_batch, b_batch, y_batch, m_batch = x_batch.to(device, non_blocking=True), b_batch.to(device, non_blocking=True), y_batch.to(device, non_blocking=True), m_batch.to(device, non_blocking=True)
                logits, chem_preds, chem_zero_mask, mole_preds, bulk_preds = FullMELTS(x_batch, binaries=b_batch, NN_only = True)
                
                # Binary saturation loss
                loss_sat = criterion_sat(logits, b_batch)
                running_sat_loss += loss_sat.item()
                
                # Chemistry losses: apply masks
                chem_loss_raw = criterion_chem(chem_preds, y_batch)
                mole_loss_raw = criterion_mole(mole_preds, m_batch)
                bulk_loss_raw = criterion_bulk(bulk_preds, x_batch[:,3:])
                bulk_zero_mask = (x_batch[:,3:] != 0).to(torch.float)
                mole_zero_mask = (torch.sigmoid(logits) > 0.5).to(torch.float).detach()
                #mole_zero_mask = (b_batch > 0.5).to(torch.float)
                
                chem_loss_masked = (chem_loss_raw * chem_zero_mask).sum() / chem_zero_mask.sum().clamp(min=1)
                mole_loss_masked = (mole_loss_raw * mole_zero_mask).sum() / mole_zero_mask.sum().clamp(min=1)
                bulk_loss_masked = (bulk_loss_raw * bulk_zero_mask).sum() / bulk_zero_mask.sum().clamp(min=1)
                
                running_mole_loss += mole_loss_masked.item()
                running_chem_loss += chem_loss_masked.item()
                running_bulk_loss += bulk_loss_masked.item()
                
                # Total loss
                #loss = loss_sat + chem_alpha*chem_loss_masked
                #loss = mole_alpha*mole_loss_masked + bulk_alpha*bulk_loss_masked
                loss = loss_sat + chem_alpha*chem_loss_masked + mole_alpha*mole_loss_masked + bulk_alpha*bulk_loss_masked
                

                running_test_loss += loss.item() * x_batch.size(0)
                if batch_idx % 50 == 0:
                    long_test_losses.append(loss.item())


        print(f"Running Saturation Loss: {round(running_sat_loss,4)}\nRunning Chem Loss: {round(running_chem_loss,4)}")
        print(f"Running Molar Loss: {round(running_mole_loss,4)}\nRunning Bulk Loss: {round(running_bulk_loss,4)}")
        
        avg_test_loss = running_test_loss / len(full_test_set)
        test_losses.append(avg_test_loss)
        if avg_test_loss <= valid_loss_min:
            torch.save(FullMELTS.state_dict(), DictFilePath)
            print('\tValidation loss decreased ({:.6f} --> {:.6f}).  Saving model ...'.format(valid_loss_min, avg_test_loss))
            valid_loss_min = avg_test_loss
            improve_record.append(1)
        else:
            improve_record.append(0)

        print(f"Epoch {epoch+1} | Train Loss: {avg_train_loss:.6f} | Test Loss: {avg_test_loss:.6f}")#/{EPOCHS}
        print(f"[TIMER] Epoch time: {time.time() - epoch_start:.2f} seconds")
        if np.sum(improve_record[-3:]) == 0:
            improve_record = [1,1]
            lr = lrs.pop()
            #if wd > 5*lr:
            #    wd = 5*lr
            print(f"No Improvement in 3 epochs. New LR: {lr}. New Weight Decay: {wd}")
            optimizer = AdamW(filter(lambda p: p.requires_grad, FullMELTS.parameters()), lr=lr, weight_decay=wd)#Adam(FullMELTS.parameters(), lr=lr)#, weight_decay=wd)
        #Adam(FullMELTS.parameters(), lr=lr)#, weight_decay=wd)
        
        #break

In [ ]:
#FULL model L2 Polishing with Bulk and custom phase weights
#torch.autograd.set_detect_anomaly(False)

weight_dict_both = {
    'olivine': 3,
    'nepheline': 15,
    'leucite': 10,
    'alloy-solid': 5,
    'muscovite': 3,
    'k-feldspar': 5
}
# This overprints the above weights, does not apply to NoCr model
weight_dict_Cr = {
    'rhm-oxide': 3
}

binWeightsNoCr = torch.ones(len(list(label_indices.keys())), dtype = torch.float32).reshape(1,-1)
binWeightsCr = torch.ones(len(list(label_indices.keys())), dtype = torch.float32).reshape(1,-1)
compWeightsNoCr = torch.ones(label_indices_comp['melts-liquid'][-1]+1, dtype = torch.float32).reshape(1,-1)
compWeightsCr = torch.ones(label_indices_comp['melts-liquid'][-1]+1, dtype = torch.float32).reshape(1,-1)
for phase, W in weight_dict_both.items():
    binWeightsCr[:,mass_phasedict[phase]] = W
    binWeightsNoCr[:,mass_phasedict[phase]] = W
    if phase in compositionally_variable_phases:
        compWeightsCr[:,comp_phasedict[phase]] = W
        compWeightsNoCr[:,comp_phasedict[phase]] = W
for phase, W in weight_dict_Cr.items():
    binWeightsCr[:,mass_phasedict[phase]] = W
    if phase in compositionally_variable_phases:
        compWeightsCr[:,comp_phasedict[phase]] = W


for i, (full_train_set, full_test_set) in enumerate([(full_train_set_NoCr, full_test_set_NoCr), (full_train_set_Cr, full_test_set_Cr)]):

    FullMELTS = DualSaturationChemistry().cuda()
    #date = "Sept30"
    #modelname = "rhyoliteMELTS1.0.2Fxtal"
    DictFilePath=f"./{modelname}{['NoCr', 'Cr'][i]}_FullL1_{date}.pt"

    FullMELTS.load_state_dict(torch.load(DictFilePath),strict = False)
    
    #date = "Sept24" 
    DictFilePath=f"./{modelname}{['NoCr', 'Cr'][i]}_FullL2_{date}.pt"
    
    
    
    batch_size = 1024

    device = 'cuda'

    binWeights = ([binWeightsNoCr, binWeightsCr][i]).to(device)
    compWeights = ([compWeightsNoCr, compWeightsCr][i]).to(device)
    
    train_loader = DataLoader(full_train_set, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True)
    test_loader = DataLoader(full_test_set, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)

    # All 
    for p in FullMELTS.parameters():
        p.requires_grad = True

        
        
    criterion_sat = torch.nn.BCEWithLogitsLoss(weight = binWeights)
    criterion_chem = symmetric_rel_l2 #relative_L1_loss #y_pred, y_true, mask=None, eps=1e-6 #criterion_chem = F.mse_loss 
    criterion_mole = symmetric_rel_l2
    criterion_bulk = symmetric_rel_l2
    
    chem_alpha = 1 #tried 1/30 last time
    mole_alpha = 1
    bulk_alpha = 0
    

    train_losses = []
    test_losses = []
    long_test_losses = []
    valid_loss_min = np.inf


    epoch = 0
    improve_record = [1,1]
    #lrs = np.logspace(-7,-3,9).tolist()
    lrs = np.logspace(-7,-4,2).tolist()
    
    lr = lrs.pop()
    wd = 1E-5
    optimizer = AdamW(FullMELTS.parameters(), lr=lr, weight_decay=wd)

    # ---- Training loop ----
    #for epoch in range(EPOCHS):
    while (lr > 2E-7):
    #while lr > 2E-25:

        epoch += 1
        epoch_start = time.time()
        FullMELTS.train()
        running_train_loss = 0.0
        total_batches = len(train_loader)

        print(f"\n--- Epoch {epoch} ---") #/{EPOCHS}

        for batch_idx, (x_batch, b_batch, y_batch, m_batch) in enumerate(tqdm(train_loader, desc="Training", leave=False)):

            optimizer.zero_grad()

            x_batch, b_batch, y_batch, m_batch = x_batch.to(device, non_blocking=True), b_batch.to(device, non_blocking=True), y_batch.to(device, non_blocking=True), m_batch.to(device, non_blocking=True)
            
            bulk_zero_mask = (x_batch != 0).to(torch.float)
            x_batch = x_batch + torch.randn_like(x_batch) * 0.003 * bulk_zero_mask # Add Small Gaussian Noise to avoid overfitting during training

            logits, chem_preds, chem_zero_mask, mole_preds, bulk_preds = FullMELTS(x_batch, binaries=b_batch, NN_only = True)

            # Binary saturation loss
            loss_sat = criterion_sat(logits, b_batch)

            # Chemistry and mass losses: apply masks
            chem_loss_raw = criterion_chem(chem_preds, y_batch)
            mole_loss_raw = criterion_mole(mole_preds, m_batch)
            bulk_loss_raw = criterion_bulk(bulk_preds, x_batch[:,3:])
            #bulk_zero_mask = (x_batch[:,3:] != 0).to(torch.float)
            mole_zero_mask = (torch.sigmoid(logits) > 0.5).to(torch.float).detach() # Only use binary preds for masking when those neurons are free
            #mole_zero_mask = (b_batch > 0.5).to(torch.float).detach()
     
            chem_loss_masked = (chem_loss_raw * chem_zero_mask * compWeights).sum() / (chem_zero_mask * compWeights).sum().clamp(min=1)
            mole_loss_masked = (mole_loss_raw * mole_zero_mask * binWeights).sum() / (mole_zero_mask * binWeights).sum().clamp(min=1)
            bulk_loss_masked = (bulk_loss_raw * bulk_zero_mask).sum() / bulk_zero_mask.sum().clamp(min=1)
            
            
            loss = loss_sat + chem_alpha*chem_loss_masked + mole_alpha*mole_loss_masked + bulk_alpha*bulk_loss_masked
            #print(f"Sat loss: {loss_sat.item()}, Chem loss: {chem_loss_masked.item()}")

            if not torch.isfinite(loss):
                print("Non-finite loss detected!")
                print(f"Sat loss: {loss_sat.item()}, Chem loss: {chem_loss_masked.item()}, Mole loss: {mole_loss_masked}, Bulk loss: {bulk_loss_masked}")
                continue

            loss.backward()


            optimizer.step()

            running_train_loss += loss.item() * x_batch.size(0)

            if batch_idx % 200 == 0:
                percent_done = 100 * batch_idx / total_batches
                train_losses.append(loss.item())
                print(f"[{percent_done:>5.1f}%] Batch {batch_idx:>5d} Loss: {loss.item():.4f}")


        avg_train_loss = running_train_loss / len(full_train_set)

        # ---- Evaluation ----
        FullMELTS.eval()
        running_test_loss = 0.0
        running_sat_loss = 0
        running_chem_loss = 0
        running_mole_loss = 0
        running_bulk_loss = 0
        
        with torch.no_grad():
            for batch_idx, (x_batch, b_batch, y_batch, m_batch) in enumerate(test_loader):
                x_batch, b_batch, y_batch, m_batch = x_batch.to(device, non_blocking=True), b_batch.to(device, non_blocking=True), y_batch.to(device, non_blocking=True), m_batch.to(device, non_blocking=True)
                logits, chem_preds, chem_zero_mask, mole_preds, bulk_preds = FullMELTS(x_batch, binaries=b_batch, NN_only = True)
                
                # Binary saturation loss
                loss_sat = criterion_sat(logits, b_batch)
                running_sat_loss += loss_sat.item()
                
                # Chemistry losses: apply masks
                chem_loss_raw = criterion_chem(chem_preds, y_batch)
                mole_loss_raw = criterion_mole(mole_preds, m_batch)
                bulk_loss_raw = criterion_bulk(bulk_preds, x_batch[:,3:])
                bulk_zero_mask = (x_batch[:,3:] != 0).to(torch.float)
                mole_zero_mask = (torch.sigmoid(logits) > 0.5).to(torch.float).detach()
                #mole_zero_mask = (b_batch > 0.5).to(torch.float)
                
                chem_loss_masked = (chem_loss_raw * chem_zero_mask).sum() / chem_zero_mask.sum().clamp(min=1)
                mole_loss_masked = (mole_loss_raw * mole_zero_mask).sum() / mole_zero_mask.sum().clamp(min=1)
                bulk_loss_masked = (bulk_loss_raw * bulk_zero_mask).sum() / bulk_zero_mask.sum().clamp(min=1)
                
                running_mole_loss += mole_loss_masked.item()
                running_chem_loss += chem_loss_masked.item()
                running_bulk_loss += bulk_loss_masked.item()
                
                # Total loss
                #loss = loss_sat + chem_alpha*chem_loss_masked
                #loss = mole_alpha*mole_loss_masked + bulk_alpha*bulk_loss_masked
                loss = loss_sat + chem_alpha*chem_loss_masked + mole_alpha*mole_loss_masked + bulk_alpha*bulk_loss_masked
                

                running_test_loss += loss.item() * x_batch.size(0)
                if batch_idx % 50 == 0:
                    long_test_losses.append(loss.item())


        print(f"Running Saturation Loss: {round(running_sat_loss,4)}\nRunning Chem Loss: {round(running_chem_loss,4)}")
        print(f"Running Molar Loss: {round(running_mole_loss,4)}\nRunning Bulk Loss: {round(running_bulk_loss,4)}")
        
        avg_test_loss = running_test_loss / len(full_test_set)
        test_losses.append(avg_test_loss)
        if avg_test_loss <= valid_loss_min:
            torch.save(FullMELTS.state_dict(), DictFilePath)
            print('\tValidation loss decreased ({:.6f} --> {:.6f}).  Saving model ...'.format(valid_loss_min, avg_test_loss))
            valid_loss_min = avg_test_loss
            improve_record.append(1)
        else:
            improve_record.append(0)

        print(f"Epoch {epoch+1} | Train Loss: {avg_train_loss:.6f} | Test Loss: {avg_test_loss:.6f}")#/{EPOCHS}
        print(f"[TIMER] Epoch time: {time.time() - epoch_start:.2f} seconds")
        if np.sum(improve_record[-3:]) == 0:
            improve_record = [1,1]
            lr = lrs.pop()
            #if wd > 5*lr:
            #    wd = 5*lr
            print(f"No Improvement in 3 epochs. New LR: {lr}. New Weight Decay: {wd}")
            optimizer = AdamW(FullMELTS.parameters(), lr=lr, weight_decay=wd)#Adam(FullMELTS.parameters(), lr=lr)#, weight_decay=wd)
        #Adam(FullMELTS.parameters(), lr=lr)#, weight_decay=wd)
        
        #break

In [ ]:
#Chem and Mole head training. NoBulk, frozen encodings
#torch.autograd.set_detect_anomaly(False)

weight_dict_both = {
    'olivine': 3,
    'nepheline': 15,
    'leucite': 10,
    'alloy-solid': 5,
    'muscovite': 3,
    'k-feldspar': 5
}
# This overprints the above weights, does not apply to NoCr model
weight_dict_Cr = {
    'rhm-oxide': 5
}

binWeightsNoCr = torch.ones(len(list(label_indices.keys())), dtype = torch.float32).reshape(1,-1)
binWeightsCr = torch.ones(len(list(label_indices.keys())), dtype = torch.float32).reshape(1,-1)
compWeightsNoCr = torch.ones(label_indices_comp['melts-liquid'][-1]+1, dtype = torch.float32).reshape(1,-1)
compWeightsCr = torch.ones(label_indices_comp['melts-liquid'][-1]+1, dtype = torch.float32).reshape(1,-1)
for phase, W in weight_dict_both.items():
    binWeightsCr[:,mass_phasedict[phase]] = W
    binWeightsNoCr[:,mass_phasedict[phase]] = W
    if phase in compositionally_variable_phases:
        compWeightsCr[:,comp_phasedict[phase]] = W
        compWeightsNoCr[:,comp_phasedict[phase]] = W
for phase, W in weight_dict_Cr.items():
    binWeightsCr[:,mass_phasedict[phase]] = W
    if phase in compositionally_variable_phases:
        compWeightsCr[:,comp_phasedict[phase]] = W


for i, (full_train_set, full_test_set) in enumerate([(full_train_set_NoCr, full_test_set_NoCr), (full_train_set_Cr, full_test_set_Cr)]):

    FullMELTS = DualSaturationChemistry().cuda()
    #date = "Sept30" 
    #modelname = "rhyoliteMELTS1.0.2Fxtal"
    DictFilePath=f"./{modelname}{['NoCr', 'Cr'][i]}_FullL2_{date}.pt"
    
    FullMELTS.load_state_dict(torch.load(DictFilePath),strict = False)
    
    #date = "Sept22"
    DictFilePath=f"./{modelname}{['NoCr', 'Cr'][i]}_Final_{date}.pt"
    
    
    
    batch_size = 1024

    device = 'cuda'

    binWeights = ([binWeightsNoCr, binWeightsCr][i]).to(device)
    compWeights = ([compWeightsNoCr, compWeightsCr][i]).to(device)
    
    train_loader = DataLoader(full_train_set, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True)
    test_loader = DataLoader(full_test_set, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)

    # All but mole heads: Put those on top.
    for p in FullMELTS.parameters():
        p.requires_grad = True
    #for p in FullMELTS.mole_head.parameters():
    #    p.requires_grad = False
    for p in FullMELTS.encoder.parameters():
        p.requires_grad = False
        
        
    criterion_sat = torch.nn.BCEWithLogitsLoss(weight = binWeights)
    criterion_chem = symmetric_rel_l2 #relative_L1_loss #y_pred, y_true, mask=None, eps=1e-6 #criterion_chem = F.mse_loss 
    criterion_mole = symmetric_rel_l2
    criterion_bulk = symmetric_rel_l2
    
    chem_alpha = 1 
    mole_alpha = 1
    bulk_alpha = 0 # For now, guassian noise in bulk
    

    train_losses = []
    test_losses = []
    long_test_losses = []
    valid_loss_min = np.inf


    epoch = 0
    improve_record = [1,1]
    lrs = np.logspace(-7,-4,2).tolist()
    lr = lrs.pop()
    wd = 1E-5
    optimizer = AdamW(filter(lambda p: p.requires_grad, FullMELTS.parameters()), lr=lr, weight_decay=wd)

    # ---- Training loop ----
    #for epoch in range(EPOCHS):
    while (lr > 2E-7):
    #while lr > 2E-25:

        epoch += 1
        epoch_start = time.time()
        FullMELTS.train()
        running_train_loss = 0.0
        total_batches = len(train_loader)

        print(f"\n--- Epoch {epoch} ---") #/{EPOCHS}

        for batch_idx, (x_batch, b_batch, y_batch, m_batch) in enumerate(tqdm(train_loader, desc="Training", leave=False)):

            optimizer.zero_grad()

            x_batch, b_batch, y_batch, m_batch = x_batch.to(device, non_blocking=True), b_batch.to(device, non_blocking=True), y_batch.to(device, non_blocking=True), m_batch.to(device, non_blocking=True)
            
            bulk_zero_mask = (x_batch != 0).to(torch.float)
            x_batch = x_batch + torch.randn_like(x_batch) * 0.002 * bulk_zero_mask # Add Small Gaussian Noise to avoid overfitting during training

            logits, chem_preds, chem_zero_mask, mole_preds, bulk_preds = FullMELTS(x_batch, binaries=b_batch, NN_only = True)

            # Binary saturation loss
            loss_sat = criterion_sat(logits, b_batch)

            # Chemistry and mass losses: apply masks
            chem_loss_raw = criterion_chem(chem_preds, y_batch)
            mole_loss_raw = criterion_mole(mole_preds, m_batch)
            bulk_loss_raw = criterion_bulk(bulk_preds, x_batch[:,3:])
            mole_zero_mask = (torch.sigmoid(logits) > 0.5).to(torch.float).detach() # Only use binary preds for masking when those neurons are free
            mole_zero_mask = (b_batch > 0.5).to(torch.float).detach()
     
            chem_loss_masked = (chem_loss_raw * chem_zero_mask * compWeights).sum() / (chem_zero_mask * compWeights).sum().clamp(min=1)
            mole_loss_masked = (mole_loss_raw * mole_zero_mask * binWeights).sum() / (mole_zero_mask * binWeights).sum().clamp(min=1)
            bulk_loss_masked = (bulk_loss_raw * bulk_zero_mask[:,3:]).sum() / (bulk_zero_mask[:,3:]).sum().clamp(min=1)
            
            
            loss = loss_sat + chem_alpha*chem_loss_masked + mole_alpha*mole_loss_masked + bulk_alpha*bulk_loss_masked
            #print(f"Sat loss: {loss_sat.item()}, Chem loss: {chem_loss_masked.item()}")

            if not torch.isfinite(loss):
                print("Non-finite loss detected!")
                print(f"Sat loss: {loss_sat.item()}, Chem loss: {chem_loss_masked.item()}, Mole loss: {mole_loss_masked}, Bulk loss: {bulk_loss_masked}")
                continue

            loss.backward()


            optimizer.step()

            running_train_loss += loss.item() * x_batch.size(0)

            if batch_idx % 200 == 0:
                percent_done = 100 * batch_idx / total_batches
                train_losses.append(loss.item())
                print(f"[{percent_done:>5.1f}%] Batch {batch_idx:>5d} Loss: {loss.item():.4f}")


        avg_train_loss = running_train_loss / len(full_train_set)

        # ---- Evaluation ----
        FullMELTS.eval()
        running_test_loss = 0.0
        running_sat_loss = 0
        running_chem_loss = 0
        running_mole_loss = 0
        running_bulk_loss = 0
        
        with torch.no_grad():
            for batch_idx, (x_batch, b_batch, y_batch, m_batch) in enumerate(test_loader):
                x_batch, b_batch, y_batch, m_batch = x_batch.to(device, non_blocking=True), b_batch.to(device, non_blocking=True), y_batch.to(device, non_blocking=True), m_batch.to(device, non_blocking=True)
                logits, chem_preds, chem_zero_mask, mole_preds, bulk_preds = FullMELTS(x_batch, binaries=b_batch, NN_only = True)
                
                # Binary saturation loss
                loss_sat = criterion_sat(logits, b_batch)
                running_sat_loss += loss_sat.item()
                
                # Chemistry losses: apply masks
                chem_loss_raw = criterion_chem(chem_preds, y_batch)
                mole_loss_raw = criterion_mole(mole_preds, m_batch)
                bulk_loss_raw = criterion_bulk(bulk_preds, x_batch[:,3:])
                bulk_zero_mask = (x_batch[:,3:] != 0).to(torch.float)
                mole_zero_mask = (torch.sigmoid(logits) > 0.5).to(torch.float).detach()
                #mole_zero_mask = (b_batch > 0.5).to(torch.float)
                
                chem_loss_masked = (chem_loss_raw * chem_zero_mask).sum() / chem_zero_mask.sum().clamp(min=1)
                mole_loss_masked = (mole_loss_raw * mole_zero_mask).sum() / mole_zero_mask.sum().clamp(min=1)
                bulk_loss_masked = (bulk_loss_raw * bulk_zero_mask).sum() / bulk_zero_mask.sum().clamp(min=1)
                
                running_mole_loss += mole_loss_masked.item()
                running_chem_loss += chem_loss_masked.item()
                running_bulk_loss += bulk_loss_masked.item()
                
                # Total loss
                #loss = loss_sat + chem_alpha*chem_loss_masked
                #loss = mole_alpha*mole_loss_masked + bulk_alpha*bulk_loss_masked
                loss = loss_sat + chem_alpha*chem_loss_masked + mole_alpha*mole_loss_masked + bulk_alpha*bulk_loss_masked
                

                running_test_loss += loss.item() * x_batch.size(0)
                if batch_idx % 50 == 0:
                    long_test_losses.append(loss.item())


        print(f"Running Saturation Loss: {round(running_sat_loss,4)}\nRunning Chem Loss: {round(running_chem_loss,4)}")
        print(f"Running Molar Loss: {round(running_mole_loss,4)}\nRunning Bulk Loss: {round(running_bulk_loss,4)}")
        
        avg_test_loss = running_test_loss / len(full_test_set)
        test_losses.append(avg_test_loss)
        if avg_test_loss <= valid_loss_min:
            torch.save(FullMELTS.state_dict(), DictFilePath)
            print('\tValidation loss decreased ({:.6f} --> {:.6f}).  Saving model ...'.format(valid_loss_min, avg_test_loss))
            valid_loss_min = avg_test_loss
            improve_record.append(1)
        else:
            improve_record.append(0)

        print(f"Epoch {epoch+1} | Train Loss: {avg_train_loss:.6f} | Test Loss: {avg_test_loss:.6f}")#/{EPOCHS}
        print(f"[TIMER] Epoch time: {time.time() - epoch_start:.2f} seconds")
        if np.sum(improve_record[-3:]) == 0:
            improve_record = [1,1]
            lr = lrs.pop()
            #if wd > 5*lr:
            #    wd = 5*lr
            print(f"No Improvement in 3 epochs. New LR: {lr}. New Weight Decay: {wd}")
            optimizer = AdamW(filter(lambda p: p.requires_grad, FullMELTS.parameters()), lr=lr, weight_decay=wd)#Adam(FullMELTS.parameters(), lr=lr)#, weight_decay=wd)
        #Adam(FullMELTS.parameters(), lr=lr)#, weight_decay=wd)
        
        #break